# 행렬 연산과 역행렬

> 선형대수 4강 · 선형방정식계와 행렬대수

이 노트북은 웹 강의의 **실습 부분만** 옮겨온 것입니다.
자세한 설명과 그림은 원문을 함께 보세요 → [행렬 연산과 역행렬](https://mioon1402.github.io/timeseriesdata/linalg/L04-matrix-ops.html)

---

**먼저 아래 준비 셀을 한 번 실행하세요.**

In [ ]:

print('준비 완료')

## 0. 기초 다지기 — 크기가 맞아야 곱할 수 있다

## 1. 전치 (Transpose)

## 2. 행렬 곱을 보는 네 가지 관점

## 3. 행렬은 변환이다

## 4. 역행렬 — 되돌리는 변환

## 5. 가우스-조던으로 역행렬 구하기

## 6. numpy 실습

**4-1. 전치와 크기**

In [ ]:
import numpy as np
np.set_printoptions(precision=3, suppress=True)

A = np.array([[1, 2, 3],
              [4, 5, 6]])

print("A 크기   :", A.shape, "  ← (행, 열)")
print("A.T 크기 :", A.T.shape)
print("\nA.T =")
print(A.T)

**4-2. 곱셈 네 가지 관점 — 전부 같은 답**

In [ ]:
B = np.array([[2, 0],
              [1, 3]])
C = np.array([[1, 4],
              [2, 5]])

print("B @ C =")
print(B @ C)
print()

print("① 원소별  (0,0) 칸 :", B[0, :] @ C[:, 0])
print("② 열 관점  첫 열   :", C[0, 0] * B[:, 0] + C[1, 0] * B[:, 1])
print("③ 행 관점  첫 행   :", B[0, 0] * C[0, :] + B[0, 1] * C[1, :])

외적합 = sum(np.outer(B[:, k], C[k, :]) for k in range(2))
print("④ 외적 합 =")
print(외적합)
print("   B@C 와 같은가:", np.array_equal(외적합, B @ C))

**4-3. 순서를 바꾸면 달라진다**

In [ ]:
print("B @ C =\n", B @ C)
print("\nC @ B =\n", C @ B)
print("\n같은가:", np.array_equal(B @ C, C @ B))
print()
print("(BC)ᵀ == CᵀBᵀ :", np.array_equal((B @ C).T, C.T @ B.T))
print("(BC)ᵀ == BᵀCᵀ :", np.array_equal((B @ C).T, B.T @ C.T), " ← 이건 틀린 식")

**4-4. * 와 @ 의 차이**

In [ ]:
print("B * C  (원소별) =")
print(B * C)
print("\nB @ C  (행렬곱) =")
print(B @ C)

**4-5. 역행렬**

In [ ]:
B_inv = np.linalg.inv(B)
print("B⁻¹ =")
print(B_inv)
print("\nB @ B⁻¹ =")
print(B @ B_inv)
print("\n단위행렬인가:", np.allclose(B @ B_inv, np.eye(2)))
print()
print("(BC)⁻¹ == C⁻¹B⁻¹ :",
      np.allclose(np.linalg.inv(B @ C), np.linalg.inv(C) @ np.linalg.inv(B)))

**4-6. 가우스-조던으로 직접 만들기**

In [ ]:
# [B | I] 를 만들어 소거한다
M = np.hstack([B.astype(float), np.eye(2)])
print("[B | I] =")
print(M)

M[0] /= M[0, 0]              # 1행 피봇을 1로
M[1] -= M[1, 0] * M[0]       # 2행에서 x 소거
M[1] /= M[1, 1]              # 2행 피봇을 1로
M[0] -= M[0, 1] * M[1]       # 1행에서 y 소거 (위쪽까지)

print("\n소거 후 =")
print(M)
print("\n오른쪽 절반이 B⁻¹ 인가:", np.allclose(M[:, 2:], B_inv))

**4-7. 특이행렬은 역행렬이 없다**

In [ ]:
특이 = np.array([[1., 2.],
                 [2., 4.]])       # 2행 = 1행의 2배

print("det =", np.linalg.det(특이))
try:
    np.linalg.inv(특이)
except np.linalg.LinAlgError as e:
    print("역행렬 없음:", e)
    print("→ 이 변환은 평면을 직선으로 눌러버려 되돌릴 수 없다")

## 7. 실무: 역행렬을 구하지 마세요

**4-8. solve vs inv — 속도와 정확도**

In [ ]:
import time

rng = np.random.default_rng(0)
n = 400
큰A = rng.random((n, n)) + n * np.eye(n)
b = rng.random(n)

t = time.perf_counter(); x1 = np.linalg.solve(큰A, b);        t1 = time.perf_counter() - t
t = time.perf_counter(); x2 = np.linalg.inv(큰A) @ b;         t2 = time.perf_counter() - t

print(f"solve   : {t1*1000:6.1f} ms   최대오차 {np.abs(큰A @ x1 - b).max():.2e}")
print(f"inv @ b : {t2*1000:6.1f} ms   최대오차 {np.abs(큰A @ x2 - b).max():.2e}")
print()
print("→ solve 가 더 빠르고 더 정확합니다.")
print("   (정확한 배수는 실행 환경에 따라 달라집니다)")

**4-9. 연습문제**

In [ ]:
# 문제 1. A = [[3, 1], [2, 4]] 의 역행렬을 2×2 공식으로 손계산하고 inv 와 비교하세요.

# 문제 2. A @ A.T 는 항상 대칭행렬입니다. 아무 A 나 만들어 확인해보세요.
#         (대칭 = 전치해도 자기 자신)

# 문제 3. [[1,2],[2,4]] 처럼 det=0 인 2×2 행렬을 하나 더 만들어보세요.
#         어떤 조건이면 det 가 0 이 되나요?

# 아래에 직접 써보세요

**모범 답안**

In [ ]:
# 문제 1
A1 = np.array([[3., 1.], [2., 4.]])
det = 3*4 - 1*2                      # = 10
손계산 = np.array([[4., -1.], [-2., 3.]]) / det
print("손계산 =\n", 손계산)
print("inv    =\n", np.linalg.inv(A1))
print("같은가:", np.allclose(손계산, np.linalg.inv(A1)))

# 문제 2
R = np.random.default_rng(1).random((3, 5))
S = R @ R.T
print("\nR@R.T 크기:", S.shape, " 대칭인가:", np.allclose(S, S.T))

# 문제 3 — 한 행(열)이 다른 행(열)의 상수배면 det = 0
for k in [3., -0.5]:
    Z = np.array([[1., 2.], [k*1., k*2.]])
    print(f"\n[[1,2],[{k},{2*k}]] det =", np.linalg.det(Z))
print("\n→ 두 행이 평행하면(한쪽이 다른 쪽의 상수배) 항상 det = 0")

---

전체 강의 목록 → [눈으로 보는 수학·통계](https://mioon1402.github.io/timeseriesdata/)